In [1]:
import torch
import torch.utils.data as data 
import torch.nn as nn
# import torch.nn.functional as F

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

## Data Loading + Processing

In [2]:
df = pd.read_csv('./data.csv', delimiter=",", index_col=0) 
df.head()

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
instant,,,,,,,,,,,,,,,,
1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [3]:
train=df.sample(frac=0.8,random_state=200) #random state is a seed value
test=df.drop(train.index)   # removes from df row indexes that are already in test dataset

In [4]:
from copy import deepcopy

input_data = deepcopy(train)
input_data = input_data.drop(columns=["casual", "registered", "cnt"])
labels = deepcopy(train["cnt"])

# print(input_data.head(), input_data.shape)
# print(labels.head())

input_data["dteday"] = pd.to_datetime(input_data["dteday"])
input_data["day"] = input_data["dteday"].dt.day
input_data.drop(columns="dteday", inplace=True)

if input_data.isna().any().any():
    for column in input_data.columns:
        input_data[column] = input_data[column].fillna(input_data[column].median())

input_data.head()

,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,day
instant,,,,,,,,,,,,,
6784,4,0,10,5,0,6,0,1,0.42,0.4242,0.71,0.1642,15
9717,1,1,2,19,0,2,1,2,0.36,0.3485,0.50,0.1642,14
3855,2,0,6,12,0,2,1,1,0.64,0.6212,0.41,0.3582,14
11262,2,1,4,10,0,4,1,1,0.48,0.4697,0.63,0.0896,19
5154,3,0,8,15,0,0,0,3,0.72,0.6970,0.74,0.2985,7


In [5]:
labels.values

array([  7, 281, 169, ..., 383,  47, 147], shape=(8709,))

In [6]:
scaler = StandardScaler()
input_data = scaler.fit_transform(input_data)
input_data

array([[ 1.34412571, -0.99827912,  1.01747441, ...,  0.46429308,
        -0.21808419,  0.9066016 ],
       [-1.34546397,  1.00172384, -1.30738054, ..., -0.63020052,
        -0.21808419,  0.72462098],
       [-0.44893408, -0.99827912, -0.14495306, ..., -1.0992692 ,
         1.36907215,  0.72462098],
       ...,
       [-0.44893408, -0.99827912, -0.7261668 , ..., -0.57808177,
         1.61369057,  0.1786791 ],
       [-0.44893408,  1.00172384, -0.43555993, ...,  0.82912427,
        -0.70650291, -1.45914652],
       [-0.44893408, -0.99827912, -0.43555993, ..., -1.88105033,
         2.46780976, -0.91320464]], shape=(8709, 13))

In [7]:
# sklearn scaler returns ndarray from transform, dataFrame.values returns ndarray of values
train_dataset = data.TensorDataset(torch.from_numpy(input_data).float(), torch.from_numpy(labels.values).float())
next(iter(train_dataset))

/var/folders/w2/3khqjd7d1m97phfx15lm1wyr0000gn/T/ipykernel_5539/2859125377.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  train_dataset = data.TensorDataset(torch.from_numpy(input_data).float(), torch.from_numpy(labels.values).float())


(tensor([ 1.3441, -0.9983,  1.0175, -0.9367, -0.1712,  1.5008, -1.4612, -0.6608,
         -0.3904, -0.2925,  0.4643, -0.2181,  0.9066]),
 tensor(7.))

In [8]:
data_loader = data.DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=False)

In [9]:
device = torch.device("mps" if torch.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

## Model definition

In [10]:
from dataclasses import dataclass


@dataclass
class HiddenLayers:
    first_layer: int = 64
    second_layer: int = 32


class Regressor(nn.Module):
    def __init__(self, input_size: int, hidden_layers: HiddenLayers = HiddenLayers()):
        super().__init__()
        self.input_size = input_size
        self.hidden_layers = hidden_layers
        self.layers = nn.Sequential(
            nn.Linear(self.input_size, self.hidden_layers.first_layer),
            nn.BatchNorm1d(self.hidden_layers.first_layer),
            nn.LeakyReLU(),
            nn.Linear(self.hidden_layers.first_layer, self.hidden_layers.second_layer),
            nn.LeakyReLU(),
            nn.Linear(self.hidden_layers.second_layer, 1),
        )

    def forward(self, input):
        return self.layers(input)


features_num = input_data.shape[1]
# features_num
model = Regressor(features_num)
model.to(device)


Regressor(
  (layers): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): LeakyReLU(negative_slope=0.01)
    (5): Linear(in_features=32, out_features=1, bias=True)
  )
)

### Train

In [11]:
EPOCHS = 300


optim = torch.optim.Adam(model.parameters(), lr=0.01)
loss_criterion = nn.MSELoss()

model.train()

for epoch in range(EPOCHS):
    for data_batch, label in data_loader:
        
        input = data_batch.to(device)
        label = label.unsqueeze(1)
        label = label.to(device)
        
        y_preds = model(input)
        loss = loss_criterion(y_preds, label)
        
        optim.zero_grad()
        loss.backward()
        optim.step()

        

### Test Data Loading + Processing

In [12]:
from copy import deepcopy

test_data = deepcopy(test)
test_data = test_data.drop(columns=["casual", "registered", "cnt"])
test_labels = deepcopy(test["cnt"])


test_data["dteday"] = pd.to_datetime(test_data["dteday"])
test_data["day"] = test_data["dteday"].dt.day
test_data.drop(columns="dteday", inplace=True)

if test_data.isna().any().any():
    for column in test_data.columns:
        test_data[column] = test_data[column].fillna(test_data[column].median())

test_data.head()

,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,day
instant,,,,,,,,,,,,,
3,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,1
4,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,1
6,1,0,1,5,0,6,0,2,0.24,0.2576,0.75,0.0896,1
18,1,0,1,17,0,6,0,2,0.44,0.4394,0.82,0.2836,1
19,1,0,1,18,0,6,0,3,0.42,0.4242,0.88,0.2537,1


In [13]:
test_data = scaler.transform(test_data)
# test_labels.values.shape

test_dataset = data.TensorDataset(torch.from_numpy(test_data).float(), torch.from_numpy(test_labels.values).float())
test_data_loader = data.DataLoader(test_dataset, batch_size=64, shuffle=True, drop_last=False)

next(iter(test_data_loader))

[tensor([[ 1.3441e+00,  1.0017e+00,  1.0175e+00,  3.6312e-01, -1.7120e-01,
           5.1519e-03,  6.8439e-01,  9.1715e-01,  3.4663e-01,  3.3398e-01,
          -9.9503e-01, -2.1808e-01,  1.2706e+00],
         [-1.3455e+00, -9.9828e-01, -1.3074e+00,  1.2297e+00, -1.7120e-01,
          -9.9192e-01,  6.8439e-01,  9.1715e-01, -9.1681e-01, -8.2918e-01,
           4.1217e-01, -7.0650e-01, -5.4924e-01],
         [-4.4893e-01,  1.0017e+00, -4.3556e-01, -6.4784e-01, -1.7120e-01,
           1.5008e+00, -1.4612e+00, -6.6078e-01, -3.9038e-01, -2.9247e-01,
           4.6429e-01, -4.6270e-01,  3.6066e-01],
         [ 4.4760e-01,  1.0017e+00,  1.4565e-01,  3.6312e-01, -1.7120e-01,
          -1.4905e+00, -1.4612e+00, -6.6078e-01,  2.4523e+00,  2.5706e+00,
          -1.3599e+00, -2.1808e-01, -3.6726e-01],
         [-1.3455e+00,  1.0017e+00, -1.3074e+00,  3.6312e-01, -1.7120e-01,
           5.0369e-01,  6.8439e-01,  2.4951e+00, -9.1681e-01, -9.1834e-01,
           9.8548e-01,  2.5716e-02,  1.0886e+00],


### Test

In [14]:
def rmsle(y_true, y_pred):
    n = len(y_true)
    msle = np.mean([(np.log(max(y_pred[i], 0) + 1) - np.log(y_true[i] + 1)) ** 2.0 for i in range(n)])
    return np.sqrt(msle)

In [15]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for test_batch, test_labels in test_data_loader:

        input = test_batch.to(device)
        test_labels = test_labels.unsqueeze(1)
        labels = test_labels.to(device)

        y_preds = model(input)

        all_preds.append(y_preds)
        all_labels.append(labels)
        
all_preds = torch.cat(all_preds).cpu().numpy()
all_labels = torch.cat(all_labels).cpu().numpy()

rmsle(all_labels, all_preds)


np.float64(0.6651066846341032)

In [16]:
eval_df = pd.read_csv('./evaluation_data.csv', delimiter=",") 
eval_df.head()

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed
0,2011-01-20,1,0,1,0,0,4,1,1,0.26,0.2273,0.56,0.3881
1,2011-01-20,1,0,1,1,0,4,1,1,0.26,0.2727,0.56,0.0000
2,2011-01-20,1,0,1,2,0,4,1,1,0.26,0.2727,0.56,0.0000
3,2011-01-20,1,0,1,3,0,4,1,1,0.26,0.2576,0.56,0.1642
4,2011-01-20,1,0,1,4,0,4,1,1,0.26,0.2576,0.56,0.1642
